# Annotation 

In [ ]:
import hail as hl
# Initialize Hail 
hl.init(default_reference = 'GRCh38')

In [ ]:
mt_v4_1 = hl.read_matrix_table("gs://ibd-exomes-gnomad-subset/QC_round4/6.final_variant_filter/gnomadv4.1.filtered.QCed.final.mt")
mt_v4_1.count()

In [ ]:
variants = mt_v4_1.rows()
variants.describe()

## gnomad nfe AF

In [ ]:
GGv4_1_ht = hl.read_table("gs://gcp-public-data--gnomad/release/4.1/ht/genomes/gnomad.genomes.v4.1.sites.ht/")
nfe_index = GGv4_1_ht.freq_index_dict['nfe_adj'].collect()[0]
variants = variants.annotate(gnomad_genomes_v4_1_nfe = GGv4_1_ht[variants.key].freq[nfe_index].AF)
# variants.gnomad_genomes_v4_1_nfe.export("gs://ibd-exomes-gnomad-subset/QC_round4/8.annotation/gnomad_genomes_v4_1_nfe.tsv.gz")

In [ ]:
gnomad_ht = hl.read_table("gs://gcp-public-data--gnomad/release/4.1/ht/genomes/gnomad.genomes.v4.1.sites.ht")

In [ ]:
freq_index_dict = gnomad_ht.freq_index_dict.collect()[0]

In [ ]:
# gnomAD v4.1 genomes HT 
gnomad_ht = hl.read_table(
    "gs://gcp-public-data--gnomad/release/4.1/ht/genomes/gnomad.genomes.v4.1.sites.ht/"
)

# get the index
freq_index_dict = gnomad_ht.freq_index_dict.collect()[0]

POPS = ["nfe"]  # "fin", "mid", "ami", "asj", "afr", "eas", "sas", "amr"

IDX = {"all": freq_index_dict["adj"]}
for pop in POPS:
    IDX[pop] = freq_index_dict[f"{pop}_adj"]

# select AF, AC, AN
freq_fields = {}  
for pop in POPS:
    freq_fields[f"AF_{pop}"] = gnomad_ht.freq[IDX[pop]].AF
    freq_fields[f"AC_{pop}"] = gnomad_ht.freq[IDX[pop]].AC
    freq_fields[f"AN_{pop}"] = gnomad_ht.freq[IDX[pop]].AN

gnomad_freq = gnomad_ht.select(**freq_fields)
#gnomad_freq.show(5)

In [ ]:
GEv4_1_ht = hl.read_table("gs://gcp-public-data--gnomad/release/4.1/ht/exomes/gnomad.exomes.v4.1.sites.ht/")
nfe_index = GEv4_1_ht.freq_index_dict['nfe_adj'].collect()[0]
variants = variants.annotate(gnomad_exomes_v4_1_nfe = GEv4_1_ht[variants.key].freq[nfe_index].AF)
# variants.gnomad_exomes_v4_1_nfe.export("gs://ibd-exomes-gnomad-subset/QC_round4/8.annotation/gnomad_exomes_v4_1_nfe.tsv.gz")

## VEP annotation

In [ ]:
#Annotating with VEP
variants_vep_anno = hl.methods.vep(variants, "gs://hail-us-central1-vep/vep95-GRCh38-loftee-gcloud.json")
variants_vep_anno.show(n_rows=5)

In [ ]:
variants_vep_anno.vep.transcript_consequences.export("gs://ibd-exomes-gnomad-subset/QC_round4/8.annotation/transcript_consequences.tsv.gz")
variants_vep_anno.vep.most_severe_consequence.export("gs://ibd-exomes-gnomad-subset/QC_round4/8.annotation/most_severe_consequences.tsv.gz")

## Other Splice

In [ ]:
os_ht = hl.read_table("gs://asc-v17/Cal_splice/20240618_GRCh38_OS.ht")
os_ht.describe()

In [ ]:
os_ht.show(n_rows=30)

In [ ]:
# If only keyed by locus (not alleles), filter instead
filtered = os_ht.filter(
    os_ht.locus == hl.locus('chr7', 99563862, reference_genome='GRCh38')
)
filtered.show()

In [ ]:
variants = variants.annotate(OS = os_ht[variants.key].OS)
variants.OS.export("gs://ibd-exomes-gnomad-subset/QC_round4/8.annotation/OS.tsv.gz")

In [ ]:
am_ht = hl.read_table("gs://nnfc-fdp-konrad-public/AlphaMissense/AlphaMissense_hg38.ht")
variants = variants.annotate(am = am_ht[variants.key])
variants = variants.filter(hl.is_defined(variants.am), keep = True)
# variants.count()
variants.am.export("gs://ibd-exomes-gnomad-subset/QC_round4/8.annotation/AlphaMissense.tsv.gz")